In [ ]:
# Parametrized notebook parameters
min_ridx = 1
max_ridx = int(1e4) 
max_cache_imgs = int(5e3)
table_id = "text_image_pairs_s2K_o10M_not_protected_20210101_20211231"

In [ ]:
gs_bucket = "example-clip-training-bucket"

In [ ]:
! gsutil mb -p example-media-project gs://{gs_bucket} && gsutil lifecycle set gcs_lifecycle_config.json gs://{gs_bucket}

In [ ]:
%pants_load media-understanding/clip/src/main/python/twitter/clip:clip_train_data_notebook

In [ ]:
import os
import tempfile

import tensorflow as tf
import tqdm
from twitter.clip.datasets.language_image_dataset import LanguageImageGcsDataset
from twitter.clip.datasets.media_cache_dataset import MediaCacheBqDataset
from twitter.clip.utils.data_utils import init_blobstore_client, write_dataset_to_gcs_tfrecords

In [ ]:
pool_size = 1000

bq_compute_project = "example-media-project"
project_id = "example-storage-project"
dataset_id = "user"

element_spec = {
  "tweet_text": tf.TensorSpec((), dtype=tf.string),
  "media_url_https": tf.TensorSpec((), dtype=tf.string),
  "tweet_id": tf.TensorSpec((), dtype=tf.int64),
}
media_field = "media_url_https"
ridx_field = "rk"

download_image_size = "small"
http_urls = False

nrecords_per_cache = 5

dataset_path = f"{table_id}_{download_image_size}_{'http' if http_urls else 'https'}"

print(dataset_path)

In [ ]:
init_blobstore_client(pool_size)

In [ ]:
for ridx in range(min_ridx, max_ridx, int(max_cache_imgs)):
    start_ridx, end_ridx = ridx, min(ridx + max_cache_imgs - 1, max_ridx)
    row_restriction = f"WHERE {ridx_field} >= {start_ridx} AND {ridx_field} <= {end_ridx}"
    prefix = f"data_ridx_{start_ridx:08}-{end_ridx:08}"

    with tempfile.TemporaryDirectory() as cache_dir:
        dataset = MediaCacheBqDataset(
            project_id=project_id,
            dataset_id=dataset_id,
            table_id=table_id,
            element_spec=element_spec,
            media_field=media_field, 
            download_image_size=download_image_size,
            http_urls=http_urls, 
            bq_compute_project=bq_compute_project,
        ).create_dataset(
            cache_dir=cache_dir,
            row_restriction=row_restriction,
            with_media_only=True,
        )
        for nsamples, features in tqdm.tqdm(enumerate(dataset), unit="samples"):
            if nsamples == 0:
              for k, v in features.items():
                print(k, v.shape)
            pass
        print(f"Found {(nsamples + 1)} samples with images.")
        samples_per_record = ((nsamples + 1) // nrecords_per_cache) + 1
        write_dataset_to_gcs_tfrecords(
            dataset=dataset,
            dataset_path=dataset_path,
            gs_bucket=gs_bucket,
            prefix=prefix,
            samples_per_record=samples_per_record,
            nsamples=(nsamples + 1)
        )

In [ ]:
# Check we can load dataset correctly
features = {
  "tweet_text": tf.io.FixedLenFeature([], tf.string),
  "image_byte_string": tf.io.FixedLenFeature([], tf.string),
  "tweet_id": tf.io.FixedLenFeature([], tf.int64),
}

dataset = LanguageImageGcsDataset(features=features).create_dataset(
  dataset_path=os.path.join("gs://", gs_bucket, dataset_path),
  is_training=False
)
for features in dataset:
  for k, v in features.items():
    print (k, v.shape, v.dtype)
  break